# ingest Json 파일 잘 만들어내는지 확인

In [1]:
import sys
from pathlib import Path

# 프로젝트 루트 경로를 Python 경로에 추가
project_root = Path.cwd().parent.parent.parent
sys.path.insert(0, str(project_root))

import requests
import xml.etree.ElementTree as ET
import json

# 모듈 리로드 (코드 수정 후 다시 테스트할 때 필요)
import importlib
from rag.etl.step01_ingest.pmc_ingest_common import pmc_parsing
importlib.reload(pmc_parsing)
from rag.etl.step01_ingest.pmc_ingest_common.pmc_parsing import extract_article_info

# PMC OAI API 엔드포인트
PMC_OAI_ENDPOINT = "https://www.ncbi.nlm.nih.gov/pmc/oai/oai.cgi"

# 테스트할 PMCID (테이블이 포함된 문서)
test_pmcid = "PMC12529775" # 테이블, 수식 이미지 확인용
# test_pmcid = "PMC11764125" # 피겨 이미지 확인용

# PMC OAI API로 문서 가져오기
params = {
    "verb": "GetRecord",
    "identifier": f"oai:pubmedcentral.nih.gov:{test_pmcid.replace('PMC', '')}",
    "metadataPrefix": "pmc"
}

print(f"테스트 문서: {test_pmcid}")
print(f"API 요청 중...")

response = requests.get(PMC_OAI_ENDPOINT, params=params, timeout=30)
response.raise_for_status()

print(f"응답 받음 (길이: {len(response.text)} bytes)")

# XML 파싱
root = ET.fromstring(response.content)

# GetRecord 응답에서 record 찾기
ns_oai = {"oai": "http://www.openarchives.org/OAI/2.0/"}
record = root.find(".//oai:record", ns_oai)

if record is None:
    print("❌ record를 찾을 수 없습니다.")
else:
    print("✓ record 찾음")
    
    # extract_article_info 호출
    article_info = extract_article_info(record)
    
    if article_info:
        print(f"\n✓ 문서 파싱 완료")
        print(f"  제목: {article_info.get('title', '')[:100]}...")
        print(f"  테이블 수: {len(article_info.get('table_captions', []))}")
        
        # 테이블 정보 출력
        for i, table in enumerate(article_info.get('table_captions', [])):
            print(f"\n=== 테이블 {i+1} ===")
            print(f"  ID: {table.get('id')}")
            print(f"  Label: {table.get('label')}")
            print(f"  Caption: {table.get('caption', '')[:100]}...")
            print(f"  Page URL: {table.get('page_url')}")
            
            # 테이블 내용 확인
            content = table.get('content')
            if content:
                print(f"\n  ✓ 테이블 내용 파싱됨!")
                
                # Structured (JSON) 확인
                structured = content.get('structured', {})
                print(f"    헤더 행 수: {len(structured.get('headers', []))}")
                print(f"    데이터 행 수: {len(structured.get('rows', []))}")
                
                # 헤더 출력
                if structured.get('headers'):
                    print(f"\n  [JSON - 헤더]")
                    for h_idx, header_row in enumerate(structured['headers']):
                        header_texts = [cell['text'] for cell in header_row]
                        print(f"    행 {h_idx+1}: {header_texts}")
                
                # 데이터 행 일부 출력 (최대 3행)
                if structured.get('rows'):
                    print(f"\n  [JSON - 데이터 행 (최대 3행)]")
                    for r_idx, row in enumerate(structured['rows'][:3]):
                        row_texts = [cell['text'] for cell in row]
                        print(f"    행 {r_idx+1}: {row_texts}")
                
                # Markdown 출력
                markdown = content.get('markdown', '')
                if markdown:
                    print(f"\n  [Markdown]")
                    print("    " + "\n    ".join(markdown.split('\n')[:6]) + "...")
                
                # Text 출력
                text = content.get('text', '')
                if text:
                    print(f"\n  [Text (검색용)]")
                    print(f"    {text[:200]}...")
                
            else:
                print(f"\n  ❌ 테이블 내용이 파싱되지 않음")
        
        # JSON 저장 (확인용)
        output_file = "test_output.json"
        with open(output_file, 'w', encoding='utf-8') as f:
            json.dump(article_info, f, indent=2, ensure_ascii=False)
        print(f"\n✓ 결과를 '{output_file}'에 저장했습니다.")
    else:
        print("❌ 문서 파싱 실패")

테스트 문서: PMC12529775
API 요청 중...
응답 받음 (길이: 108672 bytes)
✓ record 찾음

✓ 문서 파싱 완료
  제목: Unraveling the Catalytic Mechanism of β‑Cyclodextrin
in the Vitamin D Formation...
  테이블 수: 1

=== 테이블 1 ===
  ID: tbl1
  Label: table1
  Caption: Ratio Between the Rate Constant for
the Encapsulated System, [e], and the Free System, [f]...
  Page URL: https://pmc.ncbi.nlm.nih.gov/articles/PMC12529775/#tbl1

  ✓ 테이블 내용 파싱됨!
    헤더 행 수: 0
    데이터 행 수: 0

✓ 결과를 'test_output.json'에 저장했습니다.


#  2. 피겨 이미지 가져오기

## html 접근성 확인

In [2]:
from curl_cffi import requests as crequests

def test_fetch_with_impersonate(pmcid):
    url = f"https://pmc.ncbi.nlm.nih.gov/articles/{pmcid}/"
    print(f"Fetching {url} with impersonate='chrome'...")
    
    try:
        # impersonate="chrome" 옵션이 핵심입니다
        response = crequests.get(url, impersonate="chrome", timeout=30)
        print(f"Status Code: {response.status_code}")
        
        if response.status_code == 200:
            print("성공! HTML 길이:", len(response.text))
            return True
        else:
            print("실패...")
            return False
    except Exception as e:
        print(f"에러 발생: {e}")
        return False

test_fetch_with_impersonate("PMC11764125")

Fetching https://pmc.ncbi.nlm.nih.gov/articles/PMC11764125/ with impersonate='chrome'...
Status Code: 200
성공! HTML 길이: 240082


True

### 피겨 이미지 url 잘 가져오는지 확인

In [3]:
from curl_cffi import requests as crequests
from bs4 import BeautifulSoup
from pathlib import Path

def test_fetch_details(pmcid, save_html=True):
    url = f"https://pmc.ncbi.nlm.nih.gov/articles/{pmcid}/"
    print(f"Fetching {url} with impersonate='chrome'...\n")
    
    try:
        response = crequests.get(url, impersonate="chrome", timeout=30)
        
        if response.status_code == 200:
            print("✅ 요청 성공!")
            
            # HTML 파일로 저장
            if save_html:
                html_filename = f"{pmcid}_full.html"
                with open(html_filename, "w", encoding="utf-8") as f:
                    f.write(response.text)
                print(f"💾 전체 HTML을 '{html_filename}' 파일로 저장했습니다. (크기: {len(response.text):,} bytes)\n")
            
            # HTML 파싱
            soup = BeautifulSoup(response.text, "html.parser")
            
            # 1. 문서 제목 확인
            title = soup.find("title")
            print(f"📄 문서 제목: {title.text.strip() if title else '없음'}\n")
            
            # 2. 이미지 태그 찾기
            imgs = soup.find_all("img")
            print(f"🖼️ 발견된 이미지 태그 수: {len(imgs)}")
            
            # 3. Blob 이미지(우리가 찾는 것) 확인
            blob_imgs = []
            for img in imgs:
                src = img.get("src") or img.get("data-src")
                if src and "/blobs/" in src and "cdn.ncbi" in src:
                    blob_imgs.append(src)
            
            print(f"🎯 유효한 Blob 이미지 수: {len(blob_imgs)}")
            
            if blob_imgs:
                print("\n--- Blob 이미지 URL 예시 (최대 5개) ---")
                for i, url in enumerate(blob_imgs[:5]):
                    print(f"{i+1}. {url}")
            else:
                print("\n⚠️ Blob 이미지를 찾지 못했습니다. (이미지가 없는 문서일 수 있음)")
                
                # 디버깅: 일반 이미지 태그 일부 출력
                print("\n--- 일반 이미지 태그 예시 (최대 3개) ---")
                for i, img in enumerate(imgs[:3]):
                    print(f"{i+1}. {img}")

            return True
        else:
            print(f"❌ 실패: Status Code {response.status_code}")
            return False
            
    except Exception as e:
        print(f"❌ 에러 발생: {e}")
        return False

test_fetch_details("PMC11764125")

Fetching https://pmc.ncbi.nlm.nih.gov/articles/PMC11764125/ with impersonate='chrome'...

✅ 요청 성공!
💾 전체 HTML을 'PMC11764125_full.html' 파일로 저장했습니다. (크기: 240,082 bytes)

📄 문서 제목: Lipid nanoparticle delivery of TALEN mRNA targeting LPA causes gene disruption and plasma lipoprotein(a) reduction in transgenic mice - PMC

🖼️ 발견된 이미지 태그 수: 14
🎯 유효한 Blob 이미지 수: 5

--- Blob 이미지 URL 예시 (최대 5개) ---
1. https://cdn.ncbi.nlm.nih.gov/pmc/blobs/b87b/11764125/487bee00f289/fx1.jpg
2. https://cdn.ncbi.nlm.nih.gov/pmc/blobs/b87b/11764125/a9954ca2949b/gr1.jpg
3. https://cdn.ncbi.nlm.nih.gov/pmc/blobs/b87b/11764125/8f3e97c995f9/gr2.jpg
4. https://cdn.ncbi.nlm.nih.gov/pmc/blobs/b87b/11764125/22d36eb1577c/gr3.jpg
5. https://cdn.ncbi.nlm.nih.gov/pmc/blobs/b87b/11764125/59408e778ba6/gr4.jpg


True

# 3. 수식 이미지 삽입 인식하기

# 3. 수식 이미지 삽입 인식하기

## chandra - api 유료임
## 구글 비전 - 처음 1,000개 단위/월 무료
uv pip install google-cloud-vision 

## 구글 프로3 - aif에서 계정 주면 시도하기

### 구글비전 - 서비스키 발급받아서 주피터파일과 동일한 경로에 두면 됨. 서비스키 원하면 인하에게 말하세요

In [1]:
import os
from google.cloud import vision

# 서비스 계정 키 경로 설정 (한 번만 실행하면 됨)
os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = (
    "/Users/inaina/Desktop/AI/HelixOps/SKN18-FINAL-2TEAM/"
    "rag/etl/step01_ingest/adept-primer-480007-s2-e34691c5af1c.json"
)

def detect_text_uri(uri):
    """이미지 URL에서 텍스트 감지 (OCR)"""
    client = vision.ImageAnnotatorClient()
    image = vision.Image()
    image.source.image_uri = uri

    # TEXT_DETECTION (일반 텍스트) 또는 DOCUMENT_TEXT_DETECTION (문서/손글씨) 사용
    response = client.document_text_detection(image=image)
    texts = response.text_annotations

    if response.error.message:
        raise Exception(f'{response.error.message}')

    if texts:
        print(f'추출된 텍스트:\n"{texts[0].description}"')
    else:
        print("텍스트를 찾지 못했습니다.")

# 예시: 웹상의 이미지 URL
image_url = 'https://cdn.ncbi.nlm.nih.gov/pmc/blobs/039c/12529775/091c40d9be26/ci3c02049_0012.jpg'
detect_text_uri(image_url)

추출된 텍스트:
"Fel=
FPreD Fcoup
FT
coup
(4)
FB-CD"
